# 🏆 Mortgage Delinquency Risk Model
**Author:** Gurupriya R | Freddie Mac Analytics  
**Goal:** Predict mortgage delinquency using XGBoost + SHAP explainability  
**Result:** 0.83 AUC-ROC on 1,500 synthetic Freddie Mac-style loan records

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✅')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('data/loan_data.csv')
print(f'Shape: {df.shape}')
print(f'Delinquency Rate: {df["is_delinquent"].mean():.1%}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Loan Portfolio — Key Distributions', fontsize=13, fontweight='bold')
sns.histplot(df['credit_score'], ax=axes[0], color='#e8924a', bins=30)
axes[0].set_title('Credit Score Distribution')
sns.histplot(df['ltv_ratio'], ax=axes[1], color='#3a8c8c', bins=30)
axes[1].set_title('LTV Ratio Distribution')
df['is_delinquent'].value_counts().plot(kind='bar', ax=axes[2], color=['#5ab0b0','#e07060'])
axes[2].set_title('Delinquency Labels')
axes[2].set_xticklabels(['Current','Delinquent'], rotation=0)
plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
le = LabelEncoder()
for col in ['occupancy_type','loan_purpose','state']:
    df[col+'_enc'] = le.fit_transform(df[col])

df['payment_stress']    = df['dti_ratio'] * df['ltv_ratio'] / 100
df['equity_ratio']      = 100 - df['ltv_ratio']
df['risk_tier']         = pd.cut(df['credit_score'], bins=[0,580,620,680,740,850],
                                  labels=[4,3,2,1,0]).astype(int)
df['high_ltv_flag']     = (df['ltv_ratio'] > 90).astype(int)
df['loan_per_property'] = df['loan_amount'] / df['property_value']

FEATURES = [
    'credit_score','ltv_ratio','dti_ratio','loan_amount','loan_age_months',
    'original_interest_rate','num_units','num_borrowers','months_since_last_payment',
    'occupancy_type_enc','loan_purpose_enc','state_enc',
    'payment_stress','equity_ratio','risk_tier','high_ltv_flag','loan_per_property'
]
X, y = df[FEATURES], df['is_delinquent']
print('Features engineered:', len(FEATURES))

## 3. Train XGBoost Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    use_label_encoder=False, eval_metric='auc', random_state=42, verbosity=0
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('Model trained ✅')

## 4. Evaluate — AUC-ROC & Classification Report

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:,1]
y_pred = model.predict(X_test)
auc = roc_auc_score(y_test, y_pred_proba)
print(f'AUC-ROC: {auc:.4f}')
print(classification_report(y_test, y_pred, target_names=['Current','Delinquent']))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f'5-Fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Results', fontsize=13, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0].plot(fpr, tpr, color='#e8924a', lw=2.5, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve')
axes[0].legend()

feat_imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values().tail(10)
axes[1].barh(feat_imp.index, feat_imp.values, color='#e8924a')
axes[1].set_title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

## 5. SHAP Explainability

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('SHAP Feature Importance')
plt.tight_layout()
plt.show()

## 6. Flag High-Risk Borrowers & State Refinance Trends

In [ ]:
df['risk_score'] = model.predict_proba(X)[:,1]
high_risk = df[df['risk_score']>0.75][['loan_id','state','credit_score','ltv_ratio','dti_ratio','risk_score']]
high_risk = high_risk.sort_values('risk_score', ascending=False)
print(f'High-Risk Loans Flagged: {len(high_risk):,}')
high_risk.head(10)

In [ ]:
state_summary = df.groupby('state').agg(
    avg_risk_score=('risk_score','mean'),
    avg_refinance_prob=('refinance_probability','mean'),
    loan_count=('loan_id','count')
).sort_values('avg_risk_score', ascending=False)

fig, ax = plt.subplots(figsize=(12,5))
state_summary['avg_risk_score'].plot(kind='bar', ax=ax, color='#e07060', edgecolor='white')
ax.axhline(state_summary['avg_risk_score'].mean(), color='#3a8c8c', linestyle='--', label='Portfolio Avg')
ax.set_title('Average Delinquency Risk Score by State')
ax.set_ylabel('Risk Score')
ax.legend()
plt.tight_layout()
plt.show()
state_summary